# Домашнее задание: Self-Supervised + Metric Learning

**Цель:** Вам дан частично готовый ноутбук. Задача – завершить его. Вам предстоит

1. обучить CNN encoder с triplet loss и online hard-negative mining
2. извлечь frozen DINOv2 CLS-features и использовать их напрямую без обучения
4. положить embeddings в FAISS и провести validation/test оценку

## Датасет и протокол

[`nkirschi/oxford-flowers`](https://huggingface.co/datasets/nkirschi/oxford-flowers).

Выбираем 24 класса по 40 изображений (960 изображений):

- **train:** 12 классов, все 480 изображений участвуют в обучении;
- **validation:** 6 других классов, по 24 gallery + 16 query;
- **test:** ещё 6 классов, по 24 gallery + 16 query.

Множества классов train/validation/test попарно не пересекаются

In [ ]:
## Установка библиотек

%pip install -q "torch>=2.2" "torchvision>=0.17" "transformers>=4.45,<6" \
    "datasets>=3.0" "faiss-cpu>=1.8" "matplotlib>=3.8" "pillow>=10"

In [ ]:
import copy
import json
import math
import os
import random
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
import faiss
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset as TorchDataset, Sampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import transforms
from transformers import AutoModel
from typing import Generator

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    dataset_name: str = "nkirschi/oxford-flowers"
    dataset_revision: str = "a37b1891609c0376fa81eced756e7863e1bd873b"
    dino_model_name: str = "facebook/dinov2-small"
    num_classes: int = 12
    samples_per_class: int = 40
    validation_classes: int = 6
    test_classes: int = 6
    gallery_per_class: int = 24
    image_size: int = 112
    embedding_dim: int = 128
    classes_per_batch: int = 6
    samples_per_class_in_batch: int = 4
    steps_per_epoch: int = 15
    cnn_epochs: int = 16
    margin: float = 0.2
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    feature_batch_size: int = 32
    num_workers: int = 0

In [ ]:
cfg = Config()


assert 1 <= cfg.gallery_per_class < cfg.samples_per_class
assert cfg.validation_classes >= 2 and cfg.test_classes >= 2
assert 2 <= cfg.classes_per_batch <= cfg.num_classes
assert 2 <= cfg.samples_per_class_in_batch <= cfg.samples_per_class
assert cfg.image_size % 14 == 0
assert min(cfg.cnn_epochs, cfg.steps_per_epoch) > 0

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)

torch.set_num_threads(min(4, os.cpu_count() or 1))
faiss.omp_set_num_threads(1)
artifact_dir = Path("outputs/homework")
artifact_dir.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)
print("Устройство:", device)
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))


## 2. Подготовка датасета

В следующей клетке происходит:

1. Загрузка датасета с Hugging Face.
2. Выбор классов с достаточным числом изображений.
3. Разделение классов на непересекающиеся train/validation/test группы.
4. Формирование отдельные gallery/query из разных изображений.


In [ ]:
raw_dataset = load_dataset(
    cfg.dataset_name,
    split="train",
    revision=cfg.dataset_revision,
)

labels = np.asarray(raw_dataset["label"])
counts = np.bincount(labels)
eligible = np.flatnonzero(counts >= cfg.samples_per_class)
total_classes = cfg.num_classes + cfg.validation_classes + cfg.test_classes
assert len(eligible) >= total_classes
# Выбираем достаточно представленные классы, затем случайно делим их на группы.
rng = np.random.default_rng(cfg.seed)
selected_classes = np.sort(eligible[np.argsort(counts[eligible])[-total_classes:]])
class_to_new = {int(old): new for new, old in enumerate(selected_classes)}
shuffled_classes = rng.permutation(selected_classes)
class_groups = {
    "train": shuffled_classes[:cfg.num_classes],
    "validation": shuffled_classes[cfg.num_classes:cfg.num_classes + cfg.validation_classes],
    "test": shuffled_classes[cfg.num_classes + cfg.validation_classes:],
}
assert not (set(class_groups["train"]) & set(class_groups["validation"]))
assert not (set(class_groups["train"]) & set(class_groups["test"]))
assert not (set(class_groups["validation"]) & set(class_groups["test"]))
split_indices = {key: [] for key in (
    "train", "validation_gallery", "validation", "test_gallery", "test"
)}
for group, classes in class_groups.items():
    for old_class in classes:
        chosen = rng.choice(np.flatnonzero(labels == old_class), cfg.samples_per_class, replace=False)
        if group == "train":
            split_indices["train"].extend(chosen)
        else:
            split_indices[group + "_gallery"].extend(chosen[:cfg.gallery_per_class])
            split_indices[group].extend(chosen[cfg.gallery_per_class:])


def make_split(indices: list[int]) -> Dataset:
    subset = raw_dataset.select(indices)
    return subset.map(
        lambda sample: {"target": class_to_new[int(sample["label"])]},
        desc="Перенумерация классов",
    )


hf_splits = {name: make_split(indices) for name, indices in split_indices.items()}
(artifact_dir / "split_indices.json").write_text(json.dumps({
    "dataset": cfg.dataset_name,
    "revision": cfg.dataset_revision,
    "selected_classes": selected_classes.tolist(),
    "class_groups": {key: value.tolist() for key, value in class_groups.items()},
    "indices": {name: [int(i) for i in indices] for name, indices in split_indices.items()},
}, indent=2))

all_index_sets = [set(values) for values in split_indices.values()]
for i, left in enumerate(all_index_sets):
    for right in all_index_sets[i + 1:]:
        assert left.isdisjoint(right)
for split_name, split in hf_splits.items():
    split_labels = np.asarray(split["target"])
    print(split_name, len(split), "classes:", np.unique(split_labels).tolist())
for group in ("validation", "test"):
    assert set(hf_splits[group]["target"]) == set(hf_splits[group + "_gallery"]["target"])


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
sample_ids = rng.choice(len(hf_splits["train"]), size=18, replace=False)
for ax, sample_id in zip(axes.flat, sample_ids):
    sample = hf_splits["train"][int(sample_id)]
    ax.imshow(sample["image"].convert("RGB"))
    ax.set_title(f"class {sample['target']}")
    ax.axis("off")

plt.suptitle("Oxford Flowers: train subset")
plt.tight_layout()
plt.show()


## 3. Dataset wrapper, аугментации и `P × K` sampler

Batch-hard OHEM требует, чтобы в каждом batch было несколько примеров каждого класса. `P × K` sampler выбирает `P` классов и `K ≥ 2` изображений каждого класса.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(cfg.image_size, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])


class ImageDataset(TorchDataset):
    def __init__(self, dataset: Dataset, transform):
        self.dataset = dataset
        self.transform = transform
        self.labels = np.asarray(dataset["target"], dtype=np.int64)

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int):
        sample = self.dataset[index]
        image = self.transform(sample["image"].convert("RGB"))
        return image, int(sample["target"])


class PKBatchSampler(Sampler[list[int]]):
    def __init__(
        self,
        labels: np.ndarray,
        classes_per_batch: int,
        samples_per_class: int,
        steps_per_epoch: int,
        seed: int,
    ):
        self.labels = np.asarray(labels)
        self.classes_per_batch = classes_per_batch
        self.samples_per_class = samples_per_class
        self.steps_per_epoch = steps_per_epoch
        if not 2 <= classes_per_batch <= len(np.unique(self.labels)):
            raise ValueError("Нужно не менее двух классов в batch")
        counts = np.unique(self.labels, return_counts=True)[1]
        if samples_per_class < 2 or counts.min() < samples_per_class:
            raise ValueError("Нужно K >= 2 различных изображений каждого класса")
        self.seed = seed
        self.epoch = 0
        self.class_to_indices = {
            int(label): np.flatnonzero(self.labels == label)
            for label in np.unique(self.labels)
        }

    def __len__(self) -> int:
        return self.steps_per_epoch

    def __iter__(self) -> Generator[list[int], None, None]:
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        classes = np.asarray(list(self.class_to_indices))
        # Необходимо сделать генератор батчей
        # В каждом батче должно быть self.classes_per_batch классов
        # На каждый класс приходится self.samples_per_class инстансов
        # batch: list[int] – список индексов
        raise NotImplementedError("IMPLEMENT ME")

In [ ]:
train_image_dataset = ImageDataset(hf_splits["train"], train_transform)
train_eval_dataset = ImageDataset(hf_splits["train"], eval_transform)
validation_gallery_dataset = ImageDataset(hf_splits["validation_gallery"], eval_transform)
test_gallery_dataset = ImageDataset(hf_splits["test_gallery"], eval_transform)
validation_dataset = ImageDataset(hf_splits["validation"], eval_transform)
test_dataset = ImageDataset(hf_splits["test"], eval_transform)

pk_sampler = PKBatchSampler(
    train_image_dataset.labels,
    cfg.classes_per_batch,
    cfg.samples_per_class_in_batch,
    cfg.steps_per_epoch,
    cfg.seed,
)
train_loader = DataLoader(
    train_image_dataset,
    batch_sampler=pk_sampler,
    num_workers=cfg.num_workers,
)

In [ ]:
# Визуализация train-аугментаций
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
for row in range(3):
    sample = hf_splits["train"][row * cfg.samples_per_class]
    image = sample["image"].convert("RGB")
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Original, class {sample['target']}")
    for col in range(1, 5):
        view = (train_transform(image) * std + mean).clamp(0, 1)
        axes[row, col].imshow(view.permute(1, 2, 0))
        axes[row, col].set_title(f"Train view {col}")
    for ax in axes[row]:
        ax.axis("off")
plt.suptitle("Train: случайные аугментации одного изображения")
plt.tight_layout()
plt.show()

set_seed(cfg.seed)  # Визуализация не должна менять последующее обучение.

## 4. Triplet Loss и online hard example mining

Реализуйте batch-hard Triplet Loss:

- hardest positive — самый далёкий positive внутри batch;
- hardest negative — самый близкий negative внутри batch;
- anchor не должен попадать в positive mask;
- каждый anchor должен иметь хотя бы один positive и negative.

Для каждого anchor $a$ выбираем $p^* = \arg\max_{p:y_p=y_a,p\ne a}d(a,p)$
и $n^* = \arg\min_{n:y_n\ne y_a}d(a,n)$, затем усредняем
$\max(0,d(a,p^*)-d(a,n^*)+m)$ по всем anchors.
Здесь $d$ — обычное евклидово расстояние (без квадрата).
Mining выполняется заново на текущих эмбеддингах каждого batch

`active_triplets` показывает долю anchors с ненулевым loss.

In [ ]:
def pairwise_distances(embeddings: torch.Tensor) -> torch.Tensor:
    return torch.cdist(embeddings, embeddings, p=2)


def batch_hard_triplet_loss(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    margin: float,
    return_details: bool = False,
):
    distances = pairwise_distances(embeddings)
    same_class = labels[:, None] == labels[None, :]
    diagonal = torch.eye(len(labels), dtype=torch.bool, device=labels.device)

    # positive_mask: Tensor[bool] – тензор, где True — positive
    # negative_mask: Tensor[bool] – тензор, где True — negative
    positive_mask = None
    negative_mask = None

    if not positive_mask.any(dim=1).all():
        raise ValueError("У каждого anchor должен быть другой positive")
    if not negative_mask.any(dim=1).all():
        raise ValueError("У каждого anchor должен быть negative")

    # hardest_positive: Tensor[float] – тензор расстояний
    # hardest_negative: Tensor[float] – тензор расстояний
    # losses: Tensor[float] – список losses для каждого anchor
    hardest_positive = None
    hardest_negative = None
    losses = None
    loss = None

    if return_details:
        return loss, {
            "hardest_positive": hardest_positive.detach(),
            "hardest_negative": hardest_negative.detach(),
            "active_triplets": (losses > 0).float().mean().item(),
        }
    return loss


# Проверяем mining на примере с известными расстояниями и градиент.
known = torch.tensor([[0.0], [2.0], [3.0], [7.0]], requires_grad=True)
known_loss, details = batch_hard_triplet_loss(
    known, torch.tensor([0, 0, 1, 1]), margin=0.5, return_details=True
)
assert torch.allclose(details["hardest_positive"], torch.tensor([2., 2., 4., 4.]))
assert torch.allclose(details["hardest_negative"], torch.tensor([3., 1., 1., 5.]))
assert torch.allclose(known_loss, torch.tensor(1.25))
known_loss.backward()
assert torch.isfinite(known.grad).all()

# Batch без positives должен завершиться понятной ошибкой.
test_embeddings = F.normalize(torch.randn(12, 16), dim=1)
test_labels = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3])
test_loss, test_details = batch_hard_triplet_loss(
    test_embeddings, test_labels, margin=cfg.margin, return_details=True
)
assert torch.isfinite(test_loss)
assert test_details["hardest_positive"].shape == (12,)
try:
    batch_hard_triplet_loss(torch.randn(3, 8), torch.tensor([0, 1, 2]), cfg.margin)
except ValueError:
    pass
else:
    raise AssertionError("Batch без positives должен приводить к ValueError")
print("Unit tests пройдены; loss =", float(test_loss))


## 5. Модель с нуля

### Задание

Постройте небольшой residual CNN encoder (четыре residual блока) и двухслойную projection head. Обучаются оба блока;
предобученных весов нет. В projection head добавлены BatchNorm и линейные слои
без bias, чтобы уменьшить склонность к сжатию эмбеддингов в одну точку.
На выходе — L2-нормализованный embedding размерности 128.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Conv + ReLU + BN2D + Conv + ReLU + BN2D
        # Для skip-connection либо Indentity, если in_chan == out_chan,
        # либо Conv + BN2D
        raise NotImplementedError("IMPLEMENT ME")


class SmallCNN(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 24, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(24), nn.ReLU(inplace=True),
            ResidualBlock(24, 32, stride=2),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 192, bias=False), nn.BatchNorm1d(192),
            nn.ReLU(inplace=True), nn.Linear(192, embedding_dim, bias=False)
        )

    def forward(self, images):
        return F.normalize(self.head(self.encoder(images)), dim=1)


set_seed(cfg.seed)
cnn_model = SmallCNN(cfg.embedding_dim).to(device)
print(f"Параметров в SmallCNN: {sum(p.numel() for p in cnn_model.parameters()):,}")


## 6. Метрики retrieval и FAISS

---

В данном задании будет необходимо пользоваться библиотекой [faiss](https://faiss.ai/index.html).
Разобраться, как она работает, вам предстоит самостоятельно. Понадоятся:
- `faiss.FlatIPIndex`
- `faiss.IndexHNSWFlat`

---

Используем два разных понятия качества:

- **semantic Recall@K и mAP** — насколько хорошо embeddings находят изображения того же класса;
- **ANN recall@K** — насколько approximate FAISS index совпадает с exact FAISS search.

### Задание

Реализуйте метрики и точный поиск. Для query $q$ множество $R_q$ содержит все
изображения его класса в gallery. `recall@k` — среднее $|Top_k(q) \cap R_q|/|R_q|$.
При 24 релевантных изображениях максимальный `recall@1` равен 1/24.

В литературе по metric learning названием Recall@K часто обозначают **Hit@K** —
наличие хотя бы одного правильного соседа. Здесь такая метрика отдельно названа `hit@1`.

`mAP` — среднее по queries от $AP(q)=\sum_r Precision@r \cdot relevant(r)/|R_q|$;
для неё запрашиваем **всю gallery**, а не только top-10.
ANN recall@10 — доля идентификаторов из точного top-10, найденных HNSW.

In [ ]:
def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    """Нормализовать строки матрицы векторов по L2-норме (нулевые строки сохранить).

    Возвращает: np.ndarray формы (N, D), dtype float32.
    """
    vectors = np.asarray(vectors, dtype=np.float32)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True).clip(1e-12)


def build_flat_index(gallery: np.ndarray) -> faiss.IndexFlatIP:
    """Создать точный FAISS-индекс по **нормализованным** векторам gallery (G, D).

    Возвращает: IndexFlatIP с G векторами; ID — позиции строк в gallery.
    """
    raise NotImplementedError("IMPLEMENT ME")


def search_faiss(
    index: faiss.Index, queries: np.ndarray, k: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Найти k соседей для каждого нормализованного query через FAISS.

    Возвращает: (scores, ids), две матрицы формы (Q, k). scores — inner product
    (для нормализованной gallery — cosine similarity), ids — позиции в gallery.
    """
    if not 1 <= k <= index.ntotal:
        raise ValueError("k должно быть в диапазоне [1, число векторов gallery]")
    raise NotImplementedError("IMPLEMENT ME")

def semantic_metrics(
    neighbor_ids: np.ndarray,
    query_labels: np.ndarray,
    gallery_labels: np.ndarray,
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    """Посчитать retrieval-метрики по ранжированным ID и меткам классов.

    neighbor_ids имеет форму (Q, G) и содержит всю gallery в порядке поиска:
    полное ранжирование нужно для mAP. Релевантны изображения того же класса.
    Возвращает: словарь средних по queries recall@k (доля найденных релевантных),
    mAP и hit@1 (доля queries с правильным первым соседом).
    """
    result = {}
    for k in ks:
        recalls = []
        for neighbors, label in zip(neighbor_ids[:, :k], query_labels):
            relevant_total = int(np.sum(gallery_labels == label))
            relevant_found = int(np.sum(gallery_labels[neighbors] == label))
            recalls.append(relevant_found / max(relevant_total, 1))
        result[f"recall@{k}"] = float(np.mean(recalls))

    average_precisions = []
    for neighbors, label in zip(neighbor_ids, query_labels):
        raise NotImplementedError("IMPLEMENT ME")

    result["mAP"] = None
    result["hit@1"] = None

    return result


def ann_recall_at_k(exact_ids: np.ndarray, approximate_ids: np.ndarray) -> float:
    """Сравнить точный и приближённый top-k: входы имеют одинаковую форму (Q, k).
    """
    return float(np.mean([
        len(set(exact) & set(approximate)) / len(exact)
        for exact, approximate in zip(exact_ids, approximate_ids)
    ]))


def evaluate_embeddings(
    gallery_embeddings: np.ndarray,
    gallery_labels: np.ndarray,
    query_embeddings: np.ndarray,
    query_labels: np.ndarray,
) -> dict[str, float]:
    """Построить точный индекс gallery и оценить queries по полному ранжированию.

    Векторы имеют формы (G, D) и (Q, D), метки — (G,) и (Q,).
    Возвращает: словарь recall@1/5/10, mAP и hit@1 из semantic_metrics.
    """
    index = build_flat_index(gallery_embeddings)
    _, ranking = search_faiss(index, query_embeddings, k=len(gallery_embeddings))
    return semantic_metrics(ranking, query_labels, gallery_labels)


## 7. Общий training loop и выбор checkpoint

In [ ]:
@torch.no_grad()
def extract_embeddings(
    model: nn.Module, dataset: TorchDataset, batch_size: int = 64,
) -> tuple[np.ndarray, np.ndarray]:
    """Извлечь эмбеддинги в eval-режиме без градиентов, сохранив порядок dataset.

    Dataset возвращает пары (изображение или признаки, метка); модель применяется
    на глобальном device. Возвращает: (embeddings, labels), NumPy-массивы форм
    (N, D) и (N,), где N — число примеров, D — размерность выхода модели.
    """
    raise NotImplementedError("IMPLEMENT ME")


def train_metric_model(
    model: nn.Module,
    train_loader: DataLoader,
    gallery_dataset: TorchDataset,
    validation_dataset: TorchDataset,
    epochs: int,
    name: str,
    train_evaluation_dataset: TorchDataset,
) -> tuple[nn.Module, list[dict[str, float | int]]]:
    """Обучить модель с batch-hard Triplet Loss, AdamW и cosine schedule.

    train_loader содержит P×K batches. gallery_dataset и validation_dataset —
    непересекающиеся validation gallery/query; train_evaluation_dataset нужен
    для диагностики на train без аугментаций. Параметры берутся из cfg.
    Возвращает: (model, history), ту же модель с восстановленными весами лучшей
    по validation mAP эпохи и список словарей по эпохам: loss, learning rate,
    расстояния, доля активных triplets, train- и validation-метрики.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate,
                                  weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_state, best_map = None, -1.0
    history = []

    def validation_metrics(candidate):
        gallery, gy = extract_embeddings(candidate, gallery_dataset)
        queries, qy = extract_embeddings(candidate, validation_dataset)
        return evaluate_embeddings(gallery, gy, queries, qy)

    def train_metrics(candidate):
        vectors, targets = extract_embeddings(candidate, train_evaluation_dataset)
        # Фиксированные непересекающиеся gallery/query внутри train для диагностики.
        g, q = [], []
        for label in np.unique(targets):
            ids = np.flatnonzero(targets == label)
            g.extend(ids[:cfg.gallery_per_class])
            q.extend(ids[cfg.gallery_per_class:])
        return evaluate_embeddings(vectors[g], targets[g], vectors[q], targets[q])

    for epoch in range(1, epochs + 1):
        model.train()
        losses, active, positives, negatives = [], [], [], []
        current_lr = optimizer.param_groups[0]["lr"]
        for images, labels_batch in train_loader:
            # возвращайте details из batch_hard_triplet_loss
            loss, details = None, None
            raise NotImplementedError("IMPLEMENT ME")

            losses.append(loss.item())
            active.append(details["active_triplets"])
            positives.append(details["hardest_positive"].mean().item())
            negatives.append(details["hardest_negative"].mean().item())

        metrics = validation_metrics(model)
        train_result = train_metrics(model)
        scheduler.step()

        row = {
            "epoch": epoch, "loss": float(np.mean(losses)),
            "active_triplets": float(np.mean(active)), "lr": current_lr,
            "positive_distance": float(np.mean(positives)),
            "negative_distance": float(np.mean(negatives)),
            "train_mAP": train_result["mAP"], "train_hit@1": train_result["hit@1"],
            **metrics,
        }
        history.append(row)
        if metrics["mAP"] > best_map:
            best_map = metrics["mAP"]
            best_state = copy.deepcopy(model.state_dict())
        print(f"[{name}] epoch {epoch:02d}/{epochs}: loss={row['loss']:.4f}, "
              f"train mAP={row['train_mAP']:.4f}, val mAP={row['mAP']:.4f}, "
              f"best val mAP={best_map:.4f}")

    model.load_state_dict(best_state)
    print(f"{name}: лучший checkpoint, validation mAP={best_map:.4f}")
    return model, history


cnn_model, cnn_history = train_metric_model(
    cnn_model,
    train_loader,
    validation_gallery_dataset,
    validation_dataset,
    cfg.cnn_epochs,
    "CNN from scratch",
    train_eval_dataset,
)


def plot_training(history, name):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes[0, 0].plot(epochs, [row["loss"] for row in history], label="train loss")
    axes[0, 0].set_title("Batch-hard triplet loss")
    axes[0, 1].plot(epochs, [row["train_mAP"] for row in history], label="train")
    axes[0, 1].plot(epochs, [row["mAP"] for row in history], label="validation")
    axes[0, 1].set_title("mAP: виденные / новые классы")
    axes[0, 2].plot(epochs, [row["train_hit@1"] for row in history], label="train")
    axes[0, 2].plot(epochs, [row["hit@1"] for row in history], label="validation")
    axes[0, 2].set_title("Hit@1")
    axes[1, 0].plot(epochs, [row["active_triplets"] for row in history], label="active fraction")
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].set_title("Доля активных triplets")
    axes[1, 1].plot(epochs, [row["positive_distance"] for row in history], label="hard positive")
    axes[1, 1].plot(epochs, [row["negative_distance"] for row in history], label="hard negative")
    axes[1, 1].set_title("Расстояния внутри train batch")
    axes[1, 2].plot(epochs, [row["lr"] for row in history], label="learning rate")
    axes[1, 2].set_title("Learning rate")
    for ax in axes.flat:
        ax.set_xlabel("Эпоха")
        ax.legend()
        ax.grid(alpha=0.25)
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()


plot_training(cnn_history, "CNN from scratch")

## 8. Оценка CNN с нуля и построение FAISS index

### Задание

Извлеките gallery/query embeddings. Оцените точный поиск.
постройте HNSW и выберите `efSearch` на validation.

In [ ]:
cnn_gallery_embeddings, gallery_labels = extract_embeddings(cnn_model, test_gallery_dataset)
cnn_validation_gallery_embeddings, _ = extract_embeddings(cnn_model, validation_gallery_dataset)
cnn_validation_embeddings, validation_labels = extract_embeddings(
    cnn_model, validation_dataset
)
cnn_test_embeddings, test_labels = extract_embeddings(cnn_model, test_dataset)

cnn_test_metrics = evaluate_embeddings(
    cnn_gallery_embeddings, gallery_labels, cnn_test_embeddings, test_labels
)
print("CNN semantic metrics на test:", cnn_test_metrics)


def build_hnsw_index(
    gallery_embeddings: np.ndarray, m: int = 32,
) -> faiss.IndexHNSWFlat:
    """Построить HNSW-индекс по L2-нормализованной gallery формы (G, D).

    m задаёт параметр связности графа.
    Возвращает: IndexHNSWFlat с inner product и G векторами; их ID совпадают
    с позициями строк gallery.
    """
    # efConstruction фиксируйте на 100
    raise NotImplementedError("IMPLEMENT ME")


def select_hnsw_ef(
    gallery_embeddings: np.ndarray,
    validation_embeddings: np.ndarray,
    candidates: tuple[int, ...] = (8, 16, 32, 64),
    k: int = 10,
) -> tuple[faiss.IndexHNSWFlat, dict[str, float | int], list[dict[str, float | int]]]:
    """Сравнить кандидаты efSearch с точным top-k на validation queries.

    Выбирает минимальный efSearch с ANN recall >= 0.99; иначе — максимальный
    recall, затем меньший efSearch. Время — медиана десяти batch-поисков после
    warm-up, делённая на число queries; оно не участвует в выборе.
    Возвращает: (index, selected, rows), HNSW-индекс validation gallery, словарь
    выбранных efSearch/ANN recall/времени и список таких словарей для кандидатов.
    """
    flat = build_flat_index(gallery_embeddings)
    _, exact_ids = search_faiss(flat, validation_embeddings, k)
    hnsw = build_hnsw_index(gallery_embeddings)
    rows = []
    for ef_search in candidates:
        # 1. фиксируем efSearch
        # 2. Делаем warm-up (1 вызов)
        # 3. замеряем время на 10 batch-поисках после warm-up
        elapsed = None # медиана 10 batch-поисков после warm-up
        raise NotImplementedError("IMPLEMENT ME")
        rows.append({
            "efSearch": ef_search,
            "ann_recall@10": ann_recall_at_k(exact_ids, approximate_ids),
            "latency_ms_per_query": elapsed * 1e3 / len(validation_embeddings),
        })

    acceptable = [row for row in rows if row["ann_recall@10"] >= 0.99]
    selected = min(acceptable, key=lambda row: row["efSearch"]) if acceptable else max(
        rows, key=lambda row: (row["ann_recall@10"], -row["efSearch"])
    )
    return hnsw, selected, rows


cnn_hnsw, cnn_selected_hnsw, cnn_hnsw_validation = select_hnsw_ef(
    cnn_validation_gallery_embeddings, cnn_validation_embeddings
)
# Итоговый индекс содержит gallery новых test-классов; efSearch уже зафиксирован.
cnn_hnsw = build_hnsw_index(cnn_gallery_embeddings)
cnn_hnsw.hnsw.efSearch = cnn_selected_hnsw["efSearch"]
cnn_flat = build_flat_index(cnn_gallery_embeddings)
_, cnn_exact_test = search_faiss(cnn_flat, cnn_test_embeddings, 10)
_, cnn_approximate_test = search_faiss(cnn_hnsw, cnn_test_embeddings, 10)
cnn_ann_recall = ann_recall_at_k(cnn_exact_test, cnn_approximate_test)
print("Подбор HNSW на validation:", cnn_hnsw_validation)
print("Выбранная конфигурация:", cnn_selected_hnsw)
print("CNN ANN recall@10 на test:", cnn_ann_recall)


## 9. Frozen DINOv2 features

Загрузите [DINOv2-small](https://huggingface.co/facebook/dinov2-small),

отключите градиенты параметров и включите `eval()`. Используйте CLS token
последнего слоя как вектор изображения. Сохраните признаки на диск.

In [ ]:
dino = AutoModel.from_pretrained(cfg.dino_model_name).to(device)
dino.eval()
for parameter in dino.parameters():
    parameter.requires_grad_(False)


@torch.no_grad()
def extract_dino_features(
    dataset: TorchDataset, batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Извлечь CLS-признаки последнего слоя замороженного DINOv2 без градиентов.

    Использует глобальные dino (должен быть в eval-режиме) и device; dataset
    возвращает пары (подготовленное изображение, метка). Порядок сохраняется.
    Возвращает: (features, labels), CPU-тензоры форм (N, hidden_size) и (N,).
    """
    raise NotImplementedError("IMPLEMENT ME")

In [ ]:
dino_validation_gallery_features, dino_validation_gallery_labels = extract_dino_features(
    validation_gallery_dataset, cfg.feature_batch_size
)
dino_test_gallery_features, dino_test_gallery_labels = extract_dino_features(
    test_gallery_dataset, cfg.feature_batch_size
)
dino_validation_features, dino_validation_labels = extract_dino_features(
    validation_dataset, cfg.feature_batch_size
)
dino_test_features, dino_test_labels = extract_dino_features(
    test_dataset, cfg.feature_batch_size
)

torch.save({
    "validation_gallery": (dino_validation_gallery_features, dino_validation_gallery_labels),
    "test_gallery": (dino_test_gallery_features, dino_test_gallery_labels),
    "validation": (dino_validation_features, dino_validation_labels),
    "test": (dino_test_features, dino_test_labels),
    "config": asdict(cfg),
}, artifact_dir / "dino_features.pt")
assert not any(parameter.requires_grad for parameter in dino.parameters())
print("DINOv2 validation gallery features:", dino_validation_gallery_features.shape)

del dino
if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()


## 10. Подготовка raw DINOv2 embeddings

In [ ]:
dino_gallery_embeddings = l2_normalize(dino_test_gallery_features.numpy())
dino_gallery_labels = dino_test_gallery_labels.numpy()
dino_validation_gallery_embeddings = l2_normalize(dino_validation_gallery_features.numpy())
dino_validation_gallery_labels_np = dino_validation_gallery_labels.numpy()
dino_validation_embeddings = l2_normalize(dino_validation_features.numpy())
dino_validation_labels_np = dino_validation_labels.numpy()
dino_test_embeddings = l2_normalize(dino_test_features.numpy())
dino_test_labels_np = dino_test_labels.numpy()

dino_validation_metrics = evaluate_embeddings(
    dino_validation_gallery_embeddings, dino_validation_gallery_labels_np,
    dino_validation_embeddings, dino_validation_labels_np,
)
print("Raw DINOv2 validation metrics:", dino_validation_metrics)
print("Размерность raw DINOv2:", dino_gallery_embeddings.shape[1])


## 11. FAISS для DINOv2 embeddings

In [ ]:
dino_test_metrics = evaluate_embeddings(
    dino_gallery_embeddings,
    dino_gallery_labels,
    dino_test_embeddings,
    dino_test_labels_np,
)
dino_hnsw, dino_selected_hnsw, dino_hnsw_validation = select_hnsw_ef(
    dino_validation_gallery_embeddings, dino_validation_embeddings
)
dino_hnsw = build_hnsw_index(dino_gallery_embeddings)
dino_hnsw.hnsw.efSearch = dino_selected_hnsw["efSearch"]
dino_flat = build_flat_index(dino_gallery_embeddings)
_, dino_exact_test = search_faiss(dino_flat, dino_test_embeddings, 10)
_, dino_approximate_test = search_faiss(dino_hnsw, dino_test_embeddings, 10)
dino_ann_recall = ann_recall_at_k(dino_exact_test, dino_approximate_test)

index_path = artifact_dir / "dino_hnsw.faiss"
metadata_path = artifact_dir / "metadata.json"
faiss.write_index(dino_hnsw, str(index_path))
faiss.write_index(cnn_hnsw, str(artifact_dir / "cnn_hnsw.faiss"))
metadata_path.write_text(json.dumps({
    "gallery_labels": dino_gallery_labels.tolist(),
    "gallery_source_indices": [int(i) for i in split_indices["test_gallery"]],
    "dataset_revision": cfg.dataset_revision,
    "model": cfg.dino_model_name,
    "embedding_dim": int(dino_gallery_embeddings.shape[1]),
    "feature_type": "raw_cls",
    "normalization": "l2",
    "efSearch": dino_selected_hnsw["efSearch"],
    "config": asdict(cfg),
}, ensure_ascii=False, indent=2))
loaded_index = faiss.read_index(str(index_path))
_, reloaded_ids = search_faiss(loaded_index, dino_test_embeddings, 10)
assert np.array_equal(reloaded_ids, dino_approximate_test)
torch.save(cnn_model.state_dict(), artifact_dir / "cnn.pt")
print(f"Размер FAISS index: {index_path.stat().st_size / 2**20:.3f} MiB")
print("Индекс и metadata сохранены в", artifact_dir)

print("Raw DINOv2 semantic metrics на test:", dino_test_metrics)
print("Подбор HNSW на validation:", dino_hnsw_validation)
print("Выбранная конфигурация:", dino_selected_hnsw)
print("Raw DINOv2 ANN recall@10 на test:", dino_ann_recall)

## 12. Сравнение моделей

Сравниваем две модели:

1. CNN encoder, обученный с нуля;
2. raw frozen DINOv2 CLS-features без дообучения.

In [ ]:
comparison = [
    {"model": "CNN from scratch", **cnn_test_metrics, "ann_recall@10": cnn_ann_recall},
    {"model": "Raw frozen DINOv2", **dino_test_metrics, "ann_recall@10": dino_ann_recall},
]

headers = ["model", "hit@1", "recall@1", "recall@5", "recall@10", "mAP", "ann_recall@10"]
print(" | ".join(headers))
print(" | ".join(["---"] * len(headers)))
for row in comparison:
    values = [row["model"]] + [f"{row[key]:.4f}" for key in headers[1:]]
    print(" | ".join(values))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
model_names = [row["model"] for row in comparison]
axes[0].bar(model_names, [row["hit@1"] for row in comparison])
axes[0].set_title("Semantic Hit@1 на test")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(model_names, [row["mAP"] for row in comparison])
axes[1].set_title("mAP на test")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in (("CNN с нуля", cnn_history),):
    axes[0].plot([row["epoch"] for row in history], [row["loss"] for row in history], marker="o", label=name)
    axes[1].plot([row["epoch"] for row in history], [row["mAP"] for row in history], marker="o", label=name)
axes[0].set_title("Triplet Loss на train")
axes[1].set_title("mAP на validation")
for ax in axes:
    ax.set_xlabel("Эпоха")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, rows in (("CNN с нуля", cnn_hnsw_validation), ("Raw DINOv2", dino_hnsw_validation)):
    ax.plot(
        [row["latency_ms_per_query"] for row in rows],
        [row["ann_recall@10"] for row in rows], marker="o", label=name,
    )
    for row in rows:
        ax.annotate(str(row["efSearch"]), (row["latency_ms_per_query"], row["ann_recall@10"]))
ax.set_xlabel("Среднее время batch-поиска на один query, мс")
ax.set_ylabel("ANN recall@10")
ax.set_title("FAISS HNSW: настройки на validation")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

(artifact_dir / "results.json").write_text(json.dumps({
    "config": asdict(cfg),
    "comparison": comparison,
    "cnn_history": cnn_history,
    "dino_validation_metrics": dino_validation_metrics,
    "cnn_hnsw_validation": cnn_hnsw_validation,
    "dino_hnsw_validation": dino_hnsw_validation,
}, ensure_ascii=False, indent=2))


### Визуализация retrieval

Показываем один и тот же query и top-5 соседей для CNN с нуля и frozen DINOv2.

In [ ]:
query_id = 0
fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for row, (name, neighbor_ids) in enumerate((
    ("CNN с нуля", cnn_approximate_test),
    ("Raw DINOv2", dino_approximate_test),
)):
    query = hf_splits["test"][query_id]
    axes[row, 0].imshow(query["image"].convert("RGB"))
    axes[row, 0].set_title(f"{name}\nQuery: class {query['target']}")
    for rank, gallery_id in enumerate(neighbor_ids[query_id, :5], start=1):
        neighbor = hf_splits["test_gallery"][int(gallery_id)]
        axes[row, rank].imshow(neighbor["image"].convert("RGB"))
        axes[row, rank].set_title(f"#{rank}: class {neighbor['target']}")
        color = "green" if neighbor["target"] == query["target"] else "red"
        for spine in axes[row, rank].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()
plt.show()
